In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import get_project_root
print(get_project_root())
from pathlib import Path
from src.config import get_project_root

D:\MlOps_tasks\task_3


In [2]:
import pandas as pd, yaml
import numpy as np
PROJ=get_project_root()
p=yaml.safe_load(open(PROJ/"config"/'params.yaml'))
ml_table = pd.read_csv('../data/raw/ML_TABLE.csv')
print(ml_table.shape)
ml_table.head(2)

(99441, 82)


,order_id,customer_id,order_status,order_purchase,order_approved,order_delivered_carrier,order_delivered_customer,order_estimated_delivery,num_sellers,num_items,...,pay_credit_card,pay_debit_card,pay_not_defined,pay_voucher,customer_unique_id,customer_zipcode,customer_city,customer_state,customer_lat,customer_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,1.0,...,1.0,0.0,0.0,2.0,7c396fd4830fd04220f754e42b4e5bff,3149.0,sao paulo,SP,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,1.0,...,0.0,0.0,0.0,0.0,af07308b275d755c9edb36a90c618231,47813.0,barreiras,BA,-12.177924,-44.660711


In [3]:
ml_table = ml_table[ml_table['order_delivered_customer'].notna()]# delete Not deleivered orders(2965).
print(f"BEFORE \n{ml_table.dtypes.head(20)}")
date_cols = ['order_purchase','order_approved','order_delivered_carrier',
             'order_delivered_customer','order_estimated_delivery','max_shipping_limit_date']
ml_table[date_cols] = ml_table[date_cols].apply(pd.to_datetime)
print(f"AFTER \n{ml_table.dtypes.head(20)}")

####  ADD TARGET LABEL
ml_table['is_late']=np.where(ml_table['order_delivered_customer']>ml_table['order_estimated_delivery']
,1,0)
print(ml_table['is_late'].value_counts())

#################################### There is class Imbalance (92%/88649 on-time, 8%/7827 late)

BEFORE 
order_id                        str
customer_id                     str
order_status                    str
order_purchase                  str
order_approved                  str
order_delivered_carrier         str
order_delivered_customer        str
order_estimated_delivery        str
num_sellers                 float64
num_items                   float64
total_price                 float64
total_freight_value         float64
avg_seller_lat              float64
avg_seller_lng              float64
max_product_weight_grams    float64
max_product_length_cm       float64
max_product_height_cm       float64
max_product_width_cm        float64
max_shipping_limit_date         str
seller_zipcode              float64
dtype: object
AFTER 
order_id                               str
customer_id                            str
order_status                           str
order_purchase              datetime64[us]
order_approved              datetime64[us]
order_delivered_carrier     datetime

In [4]:
ml_table.head(2)

,order_id,customer_id,order_status,order_purchase,order_approved,order_delivered_carrier,order_delivered_customer,order_estimated_delivery,num_sellers,num_items,...,pay_debit_card,pay_not_defined,pay_voucher,customer_unique_id,customer_zipcode,customer_city,customer_state,customer_lat,customer_lng,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,1.0,...,0.0,0.0,2.0,7c396fd4830fd04220f754e42b4e5bff,3149.0,sao paulo,SP,-23.576983,-46.587161,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,1.0,...,0.0,0.0,0.0,af07308b275d755c9edb36a90c618231,47813.0,barreiras,BA,-12.177924,-44.660711,0


In [5]:
ml_table.to_csv(p['paths']['ml_table_labeled'], index=False)